# AlgoCourse Lecture 2: Finding an Alpha
**Imperial Algorithmic Trading Society**

Tonight we follow one trade from idea to execution and use the statistics on real data. There are five short tasks. Each one matches a slide in the lecture: have a go when the slide says **Notebook task**, then we'll show the solution.

**Before you start:** run the first code cell. It loads the price data straight from the course GitHub repo, so there's nothing to download. If you want to keep your answers, use **File → Save a copy in Drive** first.

`PAIR = "shell"` follows Shell against BP. `PAIR = "etf"` follows the US financials ETF (VFH) against four other Vanguard sector ETFs, last year's trade.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.tsa.stattools import coint, acf

PAIR = "shell"   # "shell" or "etf"

# The data lives in the course GitHub repo, so there's nothing to upload.
REPO = "https://raw.githubusercontent.com/ekaterinaveter/AlgoCourse-2026/main/"
files = {"shell": "shell_bp.csv", "etf": "vanguard_sectors.csv"}
source = files[PAIR] if os.path.exists(files[PAIR]) else REPO + files[PAIR]
prices = pd.read_csv(source, index_col="Date", parse_dates=True).dropna()
if PAIR == "shell":
    Y, X, X_MAIN = "SHEL", ["BP"], "BP"          # Y is what we trade, X is what we hedge it with
else:
    Y, X, X_MAIN = "VFH", ["VDC", "VCR", "VIS", "VGT"], "VIS"
prices.tail()

## Task 1: prices or returns?
Slides: *Puzzle 2* and *Stationarity*.

1. Plot the price of `Y` and of `X_MAIN`.
2. Compute daily **log returns**, $r_t = \ln(P_t / P_{t-1})$, for every column.
3. Compute the correlation between the two **prices**, and between their **returns**.
4. Simulate two *independent* random walks of 500 steps, $x_t = x_{t-1} + \varepsilon_t$, and compare the correlation of their levels with the correlation of their steps. Run it a few times.

Which correlation means something, and why?

In [ ]:
# 1. Plot the prices
# YOUR CODE HERE

# 2. Daily log returns for every column
returns = None  # YOUR CODE HERE

# 3. Correlation of prices and of returns
# YOUR CODE HERE

# 4. Two independent random walks
rng = np.random.default_rng()
# YOUR CODE HERE

## Task 2: autocorrelation
Slides: *Puzzle 3* and *Reading autocorrelation*.

The autocorrelation at lag $k$ is $\rho(k) = \gamma(k) / \gamma(0)$, where $\gamma(k) = \text{Cov}(x_t, x_{t-k})$.

Using 2015 to 2019 only:
1. Compute $\rho(1), \dots, \rho(10)$ for the daily returns of `Y`. (Try `acf` from statsmodels, or `Series.autocorr(lag=k)`.)
2. Do the same for the **squared** demeaned returns.
3. Plot both as bar charts. What does each one tell you?

In [ ]:
r = returns[Y].loc[:"2019"]

acf_returns = None   # YOUR CODE HERE
acf_squared = None   # YOUR CODE HERE

# Plot both
# YOUR CODE HERE

## Task 3: build the spread and test it
Slides: *How a statistical test works* and *The spread*. **Fit everything on 2015 to 2019 only.**

1. Regress log `Y` on log `X` (with a constant). The slope(s) are the hedge ratio(s) $\beta$.
2. Build the spread $s_t = \ln Y_t - c - \beta \ln X_t$ for the **whole** period, using the 2015 to 2019 fit.
3. Run the Engle-Granger cointegration test, `coint(...)`, on 2015 to 2019. Print the statistic, the 5% critical value and the p-value.
4. Half-life: regress $\Delta s_t$ on $s_{t-1}$ (with a constant). If the slope is $\gamma$, the half-life is $-\ln 2 / \ln(1+\gamma)$ days.
5. z-score: $z_t = (s_t - \bar s) / \text{sd}(s)$, with the mean and sd from 2015 to 2019. Plot $z$ for 2015 to 2019 with lines at $\pm 2$.

In [ ]:
logp = np.log(prices)
fit = logp.loc[:"2019"]

# 1. Hedge ratio(s)
model = None   # YOUR CODE HERE (hint: sm.OLS(y, sm.add_constant(x)).fit())
beta = None

# 2. Spread for the whole period
spread = None  # YOUR CODE HERE

# 3. Engle-Granger test on 2015 to 2019
# YOUR CODE HERE

# 4. Half-life
# YOUR CODE HERE

# 5. z-score and plot
z = None       # YOUR CODE HERE

## Task 4: backtest the rule on 2015 to 2019
Slides: *The trading rule* and *Backtest*.

Rule: if $z > +2$, short the spread (position $-1$); if $z < -2$, buy it ($+1$); close when $z$ crosses back through 0.

1. Write `positions(z)` that returns the position at each date's close.
2. Daily P&L per unit of spread: yesterday's position × today's change in the spread. **Why yesterday's?**
3. Charge costs: each time the position changes by 1, every leg crosses the spread, at `cost_bp` basis points per £1 traded.
4. Divide by the gross exposure, $1 + \sum|\beta|$, to get a return on the money in the trade. Report the annualised Sharpe ratio, maximum drawdown and number of trades.

*(Why no risk-free rate in this Sharpe? The short leg pays for the long leg, so no cash is tied up.)*

In [ ]:
def positions(z, entry=2.0, exit=0.0):
    pos = []
    current = 0
    for zi in z:
        # YOUR CODE HERE: update `current` using the rule
        pos.append(current)
    return pd.Series(pos, index=z.index)


def backtest(z, logp, cost_bp=5.0):
    pos = positions(z)
    # YOUR CODE HERE: pnl, costs, returns, Sharpe, max drawdown, trades
    return None


backtest(z.loc[:"2019"], logp.loc[:"2019"], cost_bp=5.0 if PAIR == "shell" else 1.0)

## Task 5: what happened next?
Slides: *Puzzle 7* and *What happened next*.

Run **exactly the same rule**, with the same hedge ratio, mean and standard deviation from 2015 to 2019, on 2020 onwards. Don't refit anything.

1. Backtest 2020 to today.
2. Plot $z$ for the whole period, 2015 to today.
3. Rerun the Engle-Granger test on 2020 to today. What changed?

In [ ]:
# 1. Same rule, data it has never seen
# YOUR CODE HERE

# 2. z-score for the whole period
# YOUR CODE HERE

# 3. Engle-Granger on 2020 onwards
# YOUR CODE HERE

## Extensions (in your own time)
- **Walk-forward:** refit $\beta$, the mean and the sd every year using only the previous two years, then trade the next year. Does it survive 2020 to today?
- **Overfitting:** try every entry level in 1.0, 1.25, ..., 3.0 and exit level in −0.5, 0, 0.5 on 2015 to 2019. Pick the best. How does it do after 2020?
- **Risk management:** add a stop-loss that closes the trade, and stops trading, once $|z| > 4$. Add a rolling two-year cointegration test and stop opening trades when its p-value goes above 0.10.
- **Switch pair:** set `PAIR = "etf"` at the top and rerun everything. Which trade would you rather have presented?